# 03 · Federated learning across substations

Federates the interpretable forecaster across feeders (Flower client + deterministic
FedAvg) and builds the **ODEON benchmark table** (federated vs centralised vs local,
plus the seasonal-naive / SARIMAX baselines) with MAPE/coverage on the named public
LV dataset. Writes `figures/odeon_benchmark.csv` and regenerates `figures/RESULTS.md`
from the same computation, so the figures, the table and the prose always agree.

In [ ]:
import os
os.environ["FORECAUS_OFFLINE"] = "1"   # run fully offline on the committed fixtures
import warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from pathlib import Path
# Resolve figures/ whether executed from the repo root or from notebooks/.
FIG = Path("notebooks/figures") if Path("notebooks").is_dir() else Path("figures")
FIG.mkdir(parents=True, exist_ok=True)
print("offline:", os.environ["FORECAUS_OFFLINE"], "| figures ->", FIG)


In [ ]:
# Single deterministic recompute (same path as scripts/reproduce_headline.py).
import sys, forecaus_grid_odeon
REPO = Path(forecaus_grid_odeon.__file__).resolve().parents[2]
sys.path.insert(0, str(REPO / 'scripts'))
import reproduce_headline as rh
head = rh.compute_headline()
c, agg_ss, fl = head['computed'], head['ss_aggregate'], head['fl_result']
print(head['table'])

In [ ]:
# ODEON benchmark table: baselines (rolling-origin) + the structured model under the
# federation protocol (local / federated-global / centralised). Protocol column keeps
# the two evaluation harnesses honestly distinct.
ROLL = 'day-ahead rolling-origin'
HOLD = 'last-day holdout, per-node'
rows = [
    {'model': 'seasonal_naive', 'role': 'baseline', 'protocol': ROLL,
     'MAPE_pct': float(agg_ss.loc['seasonal_naive', 'MAPE']), 'coverage': float(agg_ss.loc['seasonal_naive', 'coverage'])},
    {'model': 'sarimax', 'role': 'baseline', 'protocol': ROLL,
     'MAPE_pct': float(agg_ss.loc['sarimax', 'MAPE']), 'coverage': float(agg_ss.loc['sarimax', 'coverage'])},
    {'model': 'structured_gam', 'role': 'interpretable (non-federated)', 'protocol': ROLL,
     'MAPE_pct': float(agg_ss.loc['structured_gam', 'MAPE']), 'coverage': float(agg_ss.loc['structured_gam', 'coverage'])},
    {'model': 'structured_gam', 'role': 'local-only', 'protocol': HOLD, 'MAPE_pct': c['fl_local'], 'coverage': np.nan},
    {'model': 'structured_gam', 'role': 'federated-global', 'protocol': HOLD, 'MAPE_pct': c['fl_global'], 'coverage': np.nan},
    {'model': 'structured_gam', 'role': 'centralised (pooled)', 'protocol': HOLD, 'MAPE_pct': c['fl_central'], 'coverage': np.nan},
]
odeon = pd.DataFrame(rows)
odeon['MAPE_pct'] = odeon['MAPE_pct'].round(2)
odeon['coverage'] = odeon['coverage'].round(3)
odeon.insert(0, 'dataset', 'UKPN Smart Meter Consumption - LV Feeder')
odeon.to_csv(FIG / 'odeon_benchmark.csv', index=False)
print(odeon.to_string(index=False))
print('\nsaved', FIG / 'odeon_benchmark.csv')

In [ ]:
# Cold start + convergence.
print(f"cold start - thin feeder (n_train={c['cs_n_train']}): "
      f"{c['cs_local']:.2f}% local -> {c['cs_global']:.2f}% federated (+{c['cs_benefit']:.2f} pp)")
trace = fl['convergence']
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(trace) + 1), trace, marker='o', color='#4c78a8')
ax.set_xlabel('FedAvg round'); ax.set_ylabel('aggregate global MAPE [%]')
ax.set_title('Federated convergence'); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG / '03_federated_convergence.png', dpi=130); plt.close(fig)
print('saved', FIG / '03_federated_convergence.png')

In [ ]:
# Regenerate RESULTS.md from the SAME computed values and check every cited number
# traces (deterministic guard) -> the CSV, the figures and RESULTS.md agree.
from forecaus_grid_odeon.validation import validate_claims
rh.write_results_md(head['narrative'], REPO / 'notebooks' / 'figures' / 'RESULTS.md')
res = validate_claims(head['narrative'], head['computed'])
assert res['ok'], res['unsupported']
print('RESULTS.md regenerated; guard PASS -', res['n_numbers'], 'numbers all supported')